# PW 2 – Transfer Learning for Image Classification
**Master 2 Applied AI – Advanced Computer Vision (2025/2026)**

**Dataset:** *Face Mask Dataset* (binary: **without mask** vs **with mask**), from Kaggle (`shiekhburhan/face-mask-dataset`) – 500 images per class = 1000 images.

**What this notebook does (6 code cells only):**
1. Setup (imports + helper functions)
2. Load the data, split it 70 / 15 / 15, resize to 128×128 and normalize to [0, 1]
3. **Approach 1 – Feature extraction + classical ML** (SVM, Random Forest, Decision Tree, XGBoost)
4. **Approach 2 – Fine-tuning** with 5 freezing rates (0 %, 25 %, 50 %, 75 %, 100 % of layers unfrozen)
5. All the plots (training curves, bar charts, confusion matrices, ROC curves)
6. Comparative analysis + conclusions

> ⚠️ **Before running (Kaggle Notebook):**
> 1. Click **Add Input** (right panel) → search *Face Mask Dataset* by *shiekhburhan* → **Add** (no download needed).
> 2. Right panel → **Session options**: set **Accelerator = GPU (T4 / P100)** and **Internet = On** (Internet is needed to download the ImageNet weights of VGG16/InceptionV3/ResNet50).
> 3. Click **Run All**. Total running time is roughly 15–25 minutes on a GPU.
>
> (The notebook also works on Google Colab: it automatically downloads the dataset with `kagglehub` if it is not in `/kaggle/input`.)

## Cell 1 – Setup
Here we install/import all libraries and define the **global settings** and two **helper objects** used later:
* `MODELS` – the 3 pretrained CNNs we compare (VGG16, InceptionV3, ResNet50).
* `compute_metrics()` – returns accuracy, precision, recall, F1 and ROC-AUC in one call.

In [ ]:
# Kaggle (and Colab) already have TensorFlow, scikit-learn, xgboost, matplotlib installed -> nothing to pip install.

# ---------- standard libraries ----------
import os, gc, time, random            # os: files/folders | gc: free memory | time: measure duration | random: sampling
import numpy as np                      # numerical arrays
import pandas as pd                     # tables (DataFrames) and CSV files
import matplotlib.pyplot as plt         # plots
import seaborn as sns                   # nicer heatmaps (confusion matrices)
from PIL import Image                   # open and resize images

# ---------- deep learning ----------
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import VGG16, InceptionV3, ResNet50   # the 3 pretrained CNNs

# ---------- classical machine learning ----------
from sklearn.model_selection import train_test_split      # required by the homework for the 70/15/15 split
from sklearn.preprocessing import StandardScaler          # scale features (helps the SVM)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_curve, roc_auc_score)


# ---------- global settings ----------
SEED = 42                               # fixed random seed so results are reproducible
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE = 128                          # images are resized to 128x128 (allowed by the homework)
BATCH_SIZE = 32                         # number of images processed at once
EPOCHS = 20                             # maximum epochs for fine-tuning (early stopping may stop earlier)
CLASS_NAMES = ["without_mask", "with_mask"]   # label 0 = without mask, label 1 = with mask

# ---------- the 3 pretrained models we will compare ----------
MODELS = {"VGG16": VGG16, "InceptionV3": InceptionV3, "ResNet50": ResNet50}

# ---------- helper: compute every metric we need for one model ----------
def compute_metrics(y_true, y_pred, y_prob):
    # y_true = real labels, y_pred = predicted classes (0/1), y_prob = predicted probability of class 1 (needed for ROC-AUC)
    return {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall":    recall_score(y_true, y_pred),
        "F1":        f1_score(y_true, y_pred),
        "AUC":       roc_auc_score(y_true, y_prob),
    }

# Check that a GPU is available (training on CPU would be very slow) -> enable it in Session options (Kaggle) / Runtime (Colab)
print("GPU available:", tf.config.list_physical_devices("GPU"))

## Cell 2 – Data preparation
Steps performed in this cell:
1. **Find the dataset**: on Kaggle it is in `/kaggle/input` (added with *Add Input*); if it is not there (e.g. on Colab) it is downloaded with `kagglehub`.
2. **Collect** all image paths and **label them from the name of their folder** (path containing the folder `without_mask` → class 0; `with_mask` → class 1; the `incorrect_mask` folder is ignored because the task is binary). The code prints how many images were found per folder so you can check it.
3. **Load + resize** 500 random valid images per class (exactly 1000 images, as the homework asks) to 128×128 and **normalize** pixels to [0, 1] (divide by 255).
4. **Split** with `train_test_split()`: 70 % train, 15 % validation, 15 % test (stratified, so each split keeps 50/50 classes).
5. Display a few example images to check that everything is correct.

*Why this dataset?* Faces with a mask and without a mask are visually very different and the images are clean, so pretrained ImageNet features separate them very well → high, stable results.

In [ ]:
# ---------- 1. locate the dataset ----------
if os.path.exists("/kaggle/input"):                              # we are on Kaggle: the dataset was added with "Add Input"
    path = "/kaggle/input"
else:                                                            # we are on Colab (or elsewhere): download it
    import kagglehub
    path = kagglehub.dataset_download("shiekhburhan/face-mask-dataset")
print("Dataset path:", path)

# ---------- 2. collect image paths and label them from their folder name ----------
def label_from_path(folder_path):
    # In this dataset the class folder is NOT the last folder: the layout is  .../with_mask/simple/img.png
    # so we look at EVERY folder name of the path and keep the one that is exactly a class folder.
    parts = [p.lower().replace("_", "").replace("-", "").replace(" ", "") for p in folder_path.split(os.sep)]
    if "incorrectmask" in parts:  return None                    # 3rd class of the dataset (mask worn badly): ignored -> binary task
    if "withoutmask" in parts:    return 0                       # .../without_mask/...  -> class 0
    if "withmask" in parts:       return 1                       # .../with_mask/...     -> class 1
    return None                                                  # any other folder is ignored

files_by_class = {0: [], 1: []}
folder_counts = {}                                               # just to print what was found (for checking)
for root, _, files in os.walk(path):                             # walk through every sub-folder
    imgs = [os.path.join(root, f) for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    if not imgs: continue
    lab = label_from_path(root)
    folder_counts[root] = (lab, len(imgs))
    if lab is not None:
        files_by_class[lab] += imgs

print("Folders with images found:")
for folder, (lab, n) in folder_counts.items():
    print(f"  {folder}  ->  {n} images  ->  label {lab}")
print(f"Total: {len(files_by_class[0])} 'without mask' images and {len(files_by_class[1])} 'with mask' images")
assert len(files_by_class[0]) >= 500 and len(files_by_class[1]) >= 500, \
    "Could not find 2 class folders - check the printed folder names above and edit label_from_path()"

# ---------- 3. load, resize and normalize 500 valid images per class ----------
def load_image(p):
    img = Image.open(p).convert("RGB")                           # force 3 colour channels (RGB)
    img = img.resize((IMG_SIZE, IMG_SIZE))                       # resize to 128x128
    return np.asarray(img, dtype="float32") / 255.0              # normalize pixel values to [0, 1]

rng = random.Random(SEED)
X_list, y_list = [], []
for lab in [0, 1]:
    paths = files_by_class[lab]
    rng.shuffle(paths)                                           # random order -> random sample
    kept = 0
    for p in paths:
        try:
            X_list.append(load_image(p)); y_list.append(lab); kept += 1
        except Exception:
            continue                                             # skip unreadable / corrupted images
        if kept == 500: break                                    # stop at exactly 500 images for this class

X = np.stack(X_list); y = np.array(y_list)                       # final array shape: (1000, 128, 128, 3)
print("Images array:", X.shape, "| pixel range:", X.min(), "->", X.max(), "| per class:", np.bincount(y))

# ---------- 4. split 70% train / 15% validation / 15% test ----------
# first split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
# second split: the 30% is cut in half -> 15% validation + 15% test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print(f"Train: {X_train.shape[0]} | Validation: {X_val.shape[0]} | Test: {X_test.shape[0]}")

# ---------- 5. show a few examples (5 without mask, 5 with mask) ----------
idx0 = np.where(y == 0)[0][:5]; idx1 = np.where(y == 1)[0][:5]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.ravel(), list(idx0) + list(idx1)):
    ax.imshow(X[idx]); ax.set_title(CLASS_NAMES[y[idx]]); ax.axis("off")
plt.suptitle("Sample images from the dataset"); plt.tight_layout(); plt.show()

## Cell 3 – Approach 1: Feature extraction + classical ML
**Idea:** use a pretrained CNN only as a *fixed feature extractor* (no training of the CNN at all).

For each of the 3 CNNs (VGG16, InceptionV3, ResNet50):
1. Load it with ImageNet weights **without the top classification layer** (`include_top=False`).
2. **Freeze** it (`trainable = False`) so nothing is updated.
3. Pass all images through it with **global average pooling** → one flat feature vector per image (512 values for VGG16, 2048 for InceptionV3/ResNet50).
4. **Save the features + true labels in a CSV file** (`features_<model>.csv`) – required by the homework.
5. Train 4 classical classifiers on the training features: **SVM, Random Forest, Decision Tree, XGBoost**.
6. Evaluate them on the **validation + test sets together** (accuracy, precision, recall, F1, ROC-AUC) and keep the confusion matrices and ROC curves.

Finally the confusion matrices (3 models × 4 classifiers) and ROC curves are plotted.

In [ ]:
# containers where we store everything for the final comparison
ml_results = []     # one row of metrics per (CNN, classifier)
ml_cm  = {}         # confusion matrices
ml_roc = {}         # ROC curve points

# the evaluation set for the ML part = validation + test together (as requested in the homework)
y_eval = np.concatenate([y_val, y_test])

for model_name, Builder in MODELS.items():
    print(f"\n=== Extracting features with {model_name} ===")

    # --- load the pretrained CNN without its top layer; pooling="avg" adds global average pooling ---
    base = Builder(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    base.trainable = False                                   # freeze ALL layers (no backpropagation)

    # --- extract features: output is already flat (one vector per image) ---
    F_train = base.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
    F_val   = base.predict(X_val,   batch_size=BATCH_SIZE, verbose=0)
    F_test  = base.predict(X_test,  batch_size=BATCH_SIZE, verbose=0)
    print("Feature vector size per image:", F_train.shape[1])

    # --- save features + true labels (+ which split each row belongs to) in a CSV file ---
    df_feat = pd.DataFrame(np.vstack([F_train, F_val, F_test]),
                           columns=[f"f{i}" for i in range(F_train.shape[1])])
    df_feat["label"] = np.concatenate([y_train, y_val, y_test])
    df_feat["split"] = ["train"] * len(F_train) + ["val"] * len(F_val) + ["test"] * len(F_test)
    df_feat.to_csv(f"features_{model_name}.csv", index=False)   # find it in the Output / file panel

    # --- scale features (fit ONLY on train to avoid data leakage) ---
    scaler = StandardScaler().fit(F_train)
    Ftr = scaler.transform(F_train)
    Fev = scaler.transform(np.vstack([F_val, F_test]))          # validation + test features

    # --- the 4 classical classifiers ---
    classifiers = {
        "SVM":           SVC(kernel="rbf", probability=True, random_state=SEED),   # probability=True is needed for ROC-AUC
        "RandomForest":  RandomForestClassifier(n_estimators=200, random_state=SEED),
        "DecisionTree":  DecisionTreeClassifier(random_state=SEED),
        "XGBoost":       XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                       eval_metric="logloss", random_state=SEED),
    }

    for clf_name, clf in classifiers.items():
        clf.fit(Ftr, y_train)                                   # train on the training features
        y_pred = clf.predict(Fev)                               # predicted classes
        y_prob = clf.predict_proba(Fev)[:, 1]                   # probability of class "human"
        row = {"Model": model_name, "Classifier": clf_name, **compute_metrics(y_eval, y_pred, y_prob)}
        ml_results.append(row)
        ml_cm[(model_name, clf_name)] = confusion_matrix(y_eval, y_pred)
        fpr, tpr, _ = roc_curve(y_eval, y_prob)
        ml_roc[(model_name, clf_name)] = (fpr, tpr, row["AUC"])
        print(f"  {clf_name:13s} accuracy = {row['Accuracy']:.4f} | F1 = {row['F1']:.4f} | AUC = {row['AUC']:.4f}")

    del base; gc.collect(); tf.keras.backend.clear_session()    # free GPU/RAM before the next CNN

# ---------- show the results table ----------
df_ml = pd.DataFrame(ml_results)
display(df_ml.round(4))

# ---------- confusion matrices: 3 CNNs (rows) x 4 classifiers (columns) ----------
fig, axes = plt.subplots(3, 4, figsize=(16, 11))
for r, m in enumerate(MODELS):
    for c, clf_name in enumerate(["SVM", "RandomForest", "DecisionTree", "XGBoost"]):
        sns.heatmap(ml_cm[(m, clf_name)], annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[r, c])
        axes[r, c].set_title(f"{m} + {clf_name}"); axes[r, c].set_xlabel("Predicted"); axes[r, c].set_ylabel("True")
plt.suptitle("Feature extraction + ML: confusion matrices (validation + test)", fontsize=14)
plt.tight_layout(); plt.show()

# ---------- ROC curves: one plot per CNN, 4 classifiers per plot ----------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, m in zip(axes, MODELS):
    for clf_name in ["SVM", "RandomForest", "DecisionTree", "XGBoost"]:
        fpr, tpr, auc = ml_roc[(m, clf_name)]
        ax.plot(fpr, tpr, label=f"{clf_name} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--")                              # diagonal = random guessing
    ax.set_title(f"ROC curves - {m} features"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

## Cell 4 – Approach 2: Fine-tuning with different freezing rates
Here the CNN is **trained** (not only used as a feature extractor).

For each of the 3 CNNs and each of the **5 freezing rates**, the code:
1. Loads the pretrained CNN (no top layer).
2. **Unfreezes only the top X % of its layers** (X = 0, 25, 50, 75, 100). The bottom layers stay frozen because they contain generic features (edges, textures); the top layers are more task-specific.
3. Adds a new head: `GlobalAveragePooling → Dense(512) → Dropout → Dense(128) → Dense(1, sigmoid)` (sigmoid + binary cross-entropy because the task is binary).
4. Compiles with **Adam** and a small learning rate (**1e-4** when ≤ 50 % is unfrozen, **1e-5** when more is unfrozen, because updating many pretrained layers needs gentler steps).
5. Trains with **Early Stopping** (stops when validation loss stops improving and restores the best weights) and **ReduceLROnPlateau** (halves the learning rate when stuck) – these cover the "extra experiments" on early stopping / LR scheduling.
6. Evaluates on the **test set** and stores the training history for the plots.

That is 3 models × 5 rates = **15 trainings**.

In [ ]:
UNFREEZE_RATES = [0, 25, 50, 75, 100]       # % of layers (counted from the TOP of the network) that are trainable

def build_model(Builder, unfreeze_pct):
    # ---------- pretrained base (convolutional part only) ----------
    base = Builder(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    n_layers = len(base.layers)                                   # total number of layers in the base CNN
    n_frozen = int(n_layers * (1 - unfreeze_pct / 100))           # how many layers (from the bottom) stay frozen
    base.trainable = True                                         # first make everything trainable...
    for layer in base.layers[:n_frozen]:                          # ...then freeze the bottom layers
        layer.trainable = False
    if unfreeze_pct == 0:
        base.trainable = False                                    # 0% -> the whole base is frozen

    # ---------- new classification head ----------
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)                              # training=False keeps BatchNorm statistics fixed (more stable on small data)
    x = layers.GlobalAveragePooling2D()(x)                        # feature maps -> one vector
    x = layers.Dense(512, activation="relu")(x)                   # first dense layer
    x = layers.Dropout(0.3)(x)                                    # dropout reduces overfitting
    x = layers.Dense(128, activation="relu")(x)                   # second dense layer
    outputs = layers.Dense(1, activation="sigmoid")(x)            # 1 neuron + sigmoid = probability of class "human"
    model = models.Model(inputs, outputs)

    # ---------- compile: Adam + small learning rate + binary cross-entropy ----------
    lr = 1e-4 if unfreeze_pct <= 50 else 1e-5
    model.compile(optimizer=optimizers.Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
    return model

ft_results, ft_hist, ft_cm, ft_roc = [], {}, {}, {}               # containers for the final comparison

for model_name, Builder in MODELS.items():
    for pct in UNFREEZE_RATES:
        tf.keras.backend.clear_session()                          # start each run from a clean state
        model = build_model(Builder, pct)

        # callbacks: stop early if val_loss does not improve for 5 epochs; halve the LR after 2 epochs without improvement
        cbs = [callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
               callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)]

        t0 = time.time()
        hist = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                         epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=cbs, verbose=0)
        duration = time.time() - t0

        # ---------- evaluate on the TEST set ----------
        y_prob = model.predict(X_test, verbose=0).ravel()         # predicted probabilities
        y_pred = (y_prob >= 0.5).astype(int)                      # threshold 0.5 -> class 0/1
        met = compute_metrics(y_test, y_pred, y_prob)

        # overfitting gap = train accuracy - validation accuracy at the best epoch (bigger gap = more overfitting)
        h = hist.history
        best = int(np.argmin(h["val_loss"]))
        gap = h["accuracy"][best] - h["val_accuracy"][best]

        ft_results.append({"Model": model_name, "Unfreeze %": pct, **met,
                           "Overfit gap": gap, "Epochs run": len(h["loss"]), "Time (s)": round(duration)})
        ft_hist[(model_name, pct)] = h
        ft_cm[(model_name, pct)] = confusion_matrix(y_test, y_pred)
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        ft_roc[(model_name, pct)] = (fpr, tpr, met["AUC"])
        print(f"{model_name:12s} unfreeze {pct:3d}% | test acc = {met['Accuracy']:.4f} | F1 = {met['F1']:.4f} "
              f"| AUC = {met['AUC']:.4f} | epochs = {len(h['loss'])} | {duration:.0f}s")

        del model; gc.collect()

df_ft = pd.DataFrame(ft_results)
display(df_ft.round(4))

## Cell 5 – Visualizations for the fine-tuning experiments
Four groups of plots, as suggested in the homework:
1. **Training vs validation accuracy and loss** for every model and every freezing rate (one figure per CNN: top row = accuracy, bottom row = loss). A large gap between the two curves means overfitting.
2. **Bar charts** of accuracy, precision, recall and F1 for the 5 freezing rates (one chart per CNN).
3. **Confusion matrices** of all 15 fine-tuned models (test set).
4. **ROC curves** (5 freezing rates per CNN).

In [ ]:
# ---------- 1. training vs validation accuracy / loss ----------
for m in MODELS:
    fig, axes = plt.subplots(2, 5, figsize=(22, 7))
    for j, pct in enumerate(UNFREEZE_RATES):
        h = ft_hist[(m, pct)]
        # top row: accuracy
        axes[0, j].plot(h["accuracy"], label="train"); axes[0, j].plot(h["val_accuracy"], label="validation")
        axes[0, j].set_title(f"{m} - unfreeze {pct}%"); axes[0, j].set_ylabel("Accuracy"); axes[0, j].legend()
        # bottom row: loss
        axes[1, j].plot(h["loss"], label="train"); axes[1, j].plot(h["val_loss"], label="validation")
        axes[1, j].set_xlabel("Epoch"); axes[1, j].set_ylabel("Loss"); axes[1, j].legend()
    plt.suptitle(f"{m}: training vs validation (accuracy & loss)", fontsize=14)
    plt.tight_layout(); plt.show()

# ---------- 2. bar charts of the metrics for each freezing rate ----------
metrics = ["Accuracy", "Precision", "Recall", "F1"]
lo = max(0, df_ft[metrics].min().min() - 0.05)                   # lower y-limit so small differences stay visible
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, m in zip(axes, MODELS):
    df_ft[df_ft["Model"] == m].set_index("Unfreeze %")[metrics].plot(kind="bar", ax=ax, ylim=(lo, 1.02), rot=0)
    ax.set_title(f"{m}: metrics vs % of unfrozen layers"); ax.set_ylabel("Score")
plt.tight_layout(); plt.show()

# ---------- 3. confusion matrices of the 15 fine-tuned models ----------
fig, axes = plt.subplots(3, 5, figsize=(20, 11))
for r, m in enumerate(MODELS):
    for c, pct in enumerate(UNFREEZE_RATES):
        sns.heatmap(ft_cm[(m, pct)], annot=True, fmt="d", cmap="Greens", cbar=False,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[r, c])
        axes[r, c].set_title(f"{m} - unfreeze {pct}%"); axes[r, c].set_xlabel("Predicted"); axes[r, c].set_ylabel("True")
plt.suptitle("Fine-tuned models: confusion matrices (test set)", fontsize=14)
plt.tight_layout(); plt.show()

# ---------- 4. ROC curves ----------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, m in zip(axes, MODELS):
    for pct in UNFREEZE_RATES:
        fpr, tpr, auc = ft_roc[(m, pct)]
        ax.plot(fpr, tpr, label=f"unfreeze {pct}% (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--")                                # diagonal = random guessing
    ax.set_title(f"ROC curves - fine-tuned {m}"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

## Cell 6 – Comparative analysis and conclusions
This last cell puts everything together to answer the questions of the homework:
* **Feature extraction + ML vs fine-tuned CNNs** (best score of each approach per CNN)
* **VGG16 vs InceptionV3 vs ResNet50** (average and best accuracy)
* **Effect of the freezing rate** on accuracy and on overfitting (train–validation gap)
* **Binary vs multi-class:** only the binary dataset is used here (as chosen), so this comparison is not included.

Note: the ML classifiers are evaluated on validation + test (300 images) and the fine-tuned models on the test set (150 images), so differences of 1 image (≈ 0.7 %) are not really significant. All results are also saved in `all_results.csv`.

In [ ]:
# ---------- 1. best ML classifier vs best fine-tuned model, per CNN ----------
best_ml = df_ml.loc[df_ml.groupby("Model")["Accuracy"].idxmax()].set_index("Model")
best_ft = df_ft.loc[df_ft.groupby("Model")["Accuracy"].idxmax()].set_index("Model")
summary = pd.DataFrame({
    "Best ML classifier":       best_ml["Classifier"],
    "ML accuracy":              best_ml["Accuracy"],
    "Best unfreeze %":          best_ft["Unfreeze %"],
    "Fine-tuning accuracy":     best_ft["Accuracy"],
})
print("=== Feature extraction + ML  vs  Fine-tuning (best of each) ===")
display(summary.round(4))

summary[["ML accuracy", "Fine-tuning accuracy"]].plot(kind="bar", figsize=(7, 4), rot=0, ylim=(max(0, summary[["ML accuracy", "Fine-tuning accuracy"]].min().min() - 0.05), 1.02))
plt.title("Feature extraction + ML vs fine-tuning"); plt.ylabel("Accuracy"); plt.show()

# ---------- 2. average accuracy of each ML classifier (which classifier works best?) ----------
print("=== Average accuracy per classical classifier (over the 3 CNNs) ===")
display(df_ml.groupby("Classifier")[["Accuracy", "F1", "AUC"]].mean().round(4))

# ---------- 3. effect of the freezing rate (average over the 3 CNNs) ----------
print("=== Effect of the freezing rate (average over the 3 CNNs) ===")
display(df_ft.groupby("Unfreeze %")[["Accuracy", "F1", "AUC", "Overfit gap", "Time (s)"]].mean().round(4))

# ---------- 4. which pretrained CNN is better? ----------
print("=== Average results per pretrained CNN ===")
display(pd.concat([df_ml.groupby("Model")["Accuracy"].mean().rename("Mean acc (ML)"),
                   df_ft.groupby("Model")["Accuracy"].mean().rename("Mean acc (fine-tuning)")], axis=1).round(4))

# ---------- 5. overall winner + save everything ----------
top_ml = df_ml.loc[df_ml["Accuracy"].idxmax()]
top_ft = df_ft.loc[df_ft["Accuracy"].idxmax()]
print(f"Best ML      : {top_ml['Model']} + {top_ml['Classifier']}  -> accuracy {top_ml['Accuracy']:.4f}")
print(f"Best fine-tune: {top_ft['Model']} unfreeze {top_ft['Unfreeze %']}% -> accuracy {top_ft['Accuracy']:.4f}")

df_ml.assign(Approach="Feature extraction + ML").to_csv("all_results.csv", index=False)
df_ft.assign(Approach="Fine-tuning").to_csv("all_results.csv", mode="a", index=False)   # appended below the ML results
print("Saved: all_results.csv and features_<model>.csv (find them in the Output / file panel)")

## ✅ Conclusions (fill in with YOUR numbers after running)
* **Feature extraction vs fine-tuning:** which approach gave the best accuracy? Which one was faster?
* **Best pretrained model:** VGG16, InceptionV3 or ResNet50?
* **Best classical classifier:** usually SVM / XGBoost beat the single Decision Tree.
* **Freezing rate:** did unfreezing more layers help, or just increase the overfitting gap and the training time?
* **Why the scores are so high:** *Face Mask* is an easy binary problem (a mask changes the face a lot) and ImageNet features already capture such visual differences.